In [12]:
# full_analysis_script.py

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import datetime as dt
import warnings
import sys
warnings.filterwarnings("ignore")

# === Section 1: File Paths and Helpers ===
SALES_PATH = "data/monthly_sales_2023.csv"
CUSTOMERS_PATH = "data/customers.csv"
PRODUCTS_PATH = "data/products.csv"
REGIONS_PATH = "data/regions.csv"
OUTPUT_DIR = "output/"


def create_output_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Output directory created at {path}")


create_output_dir(OUTPUT_DIR)

In [13]:

# === Section 2: Data Loading ===


def load_csv(path):
    try:
        df = pd.read_csv(path)
        print(f"✅ Loaded: {path} - {df.shape}")
        return df
    except FileNotFoundError:
        print(f"❌ File not found: {path}")
        sys.exit(1)
    except pd.errors.EmptyDataError:
        print(f"❌ Empty file: {path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ Unknown error: {e}")
        return pd.DataFrame()


sales_df = load_csv(SALES_PATH)
cust_df = load_csv(CUSTOMERS_PATH)
prod_df = load_csv(PRODUCTS_PATH)
region_df = load_csv(REGIONS_PATH)

✅ Loaded: data/monthly_sales_2023.csv - (2000, 5)
✅ Loaded: data/customers.csv - (100, 5)
✅ Loaded: data/products.csv - (50, 4)
✅ Loaded: data/regions.csv - (5, 2)


In [14]:

# === Section 3: Initial Cleaning ===


def clean_cols(df):
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
    return df


sales_df = clean_cols(sales_df)
cust_df = clean_cols(cust_df)
prod_df = clean_cols(prod_df)
region_df = clean_cols(region_df)

# Drop bad rows
sales_df = sales_df.dropna(subset=["order_id", "customer_id", "product_id"])
sales_df = sales_df[sales_df['order_value'] > 0]

# Convert types
try:
    sales_df['order_date'] = pd.to_datetime(sales_df['order_date'])
except Exception as e:
    print("⚠️ Failed to parse dates in order_date:", e)

In [15]:
sales_df

,order_id,customer_id,product_id,order_date,order_value
0,a7b1086a-5d7f-4e5a-91cd-1d7495ce5fb7,1072,505,2025-04-07,1667.24
1,b70dc54f-8060-460a-be94-2c9e9ca49595,1074,532,2025-04-07,377.96
2,43dcbcbb-fb62-4f65-be43-ca138818b590,1064,532,2025-04-07,1543.43
3,2b94a336-d529-4145-a4db-e8f06ec6475b,1100,550,2025-04-07,1898.01
4,14d1d90a-ad98-4222-ae61-a76240b78071,1091,522,2025-04-07,717.17
...,...,...,...,...,...
1995,155a247f-56dd-43f8-a0c9-cf26449c5bc0,1016,509,2025-04-07,136.38
1996,f1c44649-5adc-43d3-8919-044c38c71cd7,1030,535,2025-04-07,181.40
1997,91fa2dff-65c2-45df-84e9-c2b92c8599fc,1069,514,2025-04-07,1425.08
1998,1e292b79-49e9-4f28-9e94-1a537750f64c,1079,526,2025-04-07,435.84


In [16]:
df1 = pd.merge(sales_df, cust_df, on='customer_id', how='left')

In [17]:
df1

,order_id,customer_id,product_id,order_date,order_value,name,age,email,region_id
0,a7b1086a-5d7f-4e5a-91cd-1d7495ce5fb7,1072,505,2025-04-07,1667.24,Brandon Johnson,54,psherman@example.net,3
1,b70dc54f-8060-460a-be94-2c9e9ca49595,1074,532,2025-04-07,377.96,Jennifer Hayes,41,fitzgeraldbobby@example.com,1
2,43dcbcbb-fb62-4f65-be43-ca138818b590,1064,532,2025-04-07,1543.43,Sierra Mendoza,27,johnsonyolanda@example.org,5
3,2b94a336-d529-4145-a4db-e8f06ec6475b,1100,550,2025-04-07,1898.01,Adam George,31,jross@example.net,3
4,14d1d90a-ad98-4222-ae61-a76240b78071,1091,522,2025-04-07,717.17,Paula Ball,65,coxmindy@example.org,1
...,...,...,...,...,...,...,...,...,...
1995,155a247f-56dd-43f8-a0c9-cf26449c5bc0,1016,509,2025-04-07,136.38,Jonathan Martin,37,nancy72@example.net,3
1996,f1c44649-5adc-43d3-8919-044c38c71cd7,1030,535,2025-04-07,181.40,Jeremiah Price,44,meganterry@example.net,4
1997,91fa2dff-65c2-45df-84e9-c2b92c8599fc,1069,514,2025-04-07,1425.08,Daniel Marshall,35,erica88@example.com,5
1998,1e292b79-49e9-4f28-9e94-1a537750f64c,1079,526,2025-04-07,435.84,Nicholas Welch,30,carolyn18@example.org,1


In [18]:
df2 = pd.merge(df1, prod_df, on='product_id', how='left')

In [19]:
df2

,order_id,customer_id,product_id,order_date,order_value,name_x,age,email,region_id,name_y,category,price
0,a7b1086a-5d7f-4e5a-91cd-1d7495ce5fb7,1072,505,2025-04-07,1667.24,Brandon Johnson,54,psherman@example.net,3,Old,Fashion,314.83
1,b70dc54f-8060-460a-be94-2c9e9ca49595,1074,532,2025-04-07,377.96,Jennifer Hayes,41,fitzgeraldbobby@example.com,1,Recently,Home,192.81
2,43dcbcbb-fb62-4f65-be43-ca138818b590,1064,532,2025-04-07,1543.43,Sierra Mendoza,27,johnsonyolanda@example.org,5,Recently,Home,192.81
3,2b94a336-d529-4145-a4db-e8f06ec6475b,1100,550,2025-04-07,1898.01,Adam George,31,jross@example.net,3,Word,Fashion,77.28
4,14d1d90a-ad98-4222-ae61-a76240b78071,1091,522,2025-04-07,717.17,Paula Ball,65,coxmindy@example.org,1,Process,Books,444.57
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,155a247f-56dd-43f8-a0c9-cf26449c5bc0,1016,509,2025-04-07,136.38,Jonathan Martin,37,nancy72@example.net,3,Where,Books,100.38
1996,f1c44649-5adc-43d3-8919-044c38c71cd7,1030,535,2025-04-07,181.40,Jeremiah Price,44,meganterry@example.net,4,Significant,Home,118.67
1997,91fa2dff-65c2-45df-84e9-c2b92c8599fc,1069,514,2025-04-07,1425.08,Daniel Marshall,35,erica88@example.com,5,Little,Books,133.12
1998,1e292b79-49e9-4f28-9e94-1a537750f64c,1079,526,2025-04-07,435.84,Nicholas Welch,30,carolyn18@example.org,1,Plan,Electronics,40.78


In [20]:
df3 = pd.merge(df2, region_df, on='region_id', how='left')

In [5]:

# === Section 4: Enriching ===


def enrich_data(sales):
    try:
        df1 = pd.merge(sales, cust_df, on='customer_id', how='left')
        df2 = pd.merge(df1, prod_df, on='product_id', how='left')
        df3 = pd.merge(df2, region_df, on='region_id', how='left')
        return df3
    except Exception as e:
        print("⚠️ Data enrichment failed:", e)
        return sales


full_df = enrich_data(sales_df)

# Add new fields
full_df['year'] = full_df['order_date'].dt.year
full_df['month'] = full_df['order_date'].dt.month
full_df['is_high_value'] = full_df['order_value'] > 1000

⚠️ Data enrichment failed: 'region_id'
